In [1]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = Path(os.environ.get("NEISS_DATA_DIR", Path.home() / "Downloads" / "NEISS Data" / "NEISS Data")).resolve()
BASE_OUTPUT_DIR = PROJECT_DIR / "outputs" / "neiss_world_cup_v2"
REVISION_OUTPUT_DIR = PROJECT_DIR / "outputs" / "professor_revision_20260620"
RAW_FILES = [DATA_DIR / f"neiss{year}.xlsx" for year in range(1999, 2026)]
CACHE_FILES = [BASE_OUTPUT_DIR / "cache" / "yearly" / f"psu_day_{year}.parquet" for year in range(1999, 2026)]

missing_raw = [str(path) for path in RAW_FILES if not path.exists()]
missing_cache = [str(path) for path in CACHE_FILES if not path.exists()]
if missing_raw:
    raise FileNotFoundError(f"Missing annual NEISS workbooks: {missing_raw}")
if missing_cache:
    raise FileNotFoundError(f"Missing yearly PSU-day caches: {missing_cache}")

print(f"Project directory: {PROJECT_DIR}")
print(f"NEISS data directory: {DATA_DIR}")
print(f"Annual workbooks verified: {len(RAW_FILES)}")
print(f"Yearly PSU-day caches verified: {len(CACHE_FILES)}")


Project directory: D:\ai-job-agent\MIMIC_Project_3
NEISS data directory: C:\Users\rendu\Downloads\NEISS Data\NEISS Data
Annual workbooks verified: 27
Yearly PSU-day caches verified: 27


In [2]:
environment = dict(os.environ)
environment["NEISS_DATA_DIR"] = str(DATA_DIR)
environment["MPLBACKEND"] = "Agg"
environment["MPLCONFIGDIR"] = str(PROJECT_DIR / ".matplotlib")
(PROJECT_DIR / ".matplotlib").mkdir(parents=True, exist_ok=True)

base_run = subprocess.run(
    [sys.executable, str(PROJECT_DIR / "analysis_v2" / "neiss_world_cup_analysis_v2.py")],
    cwd=PROJECT_DIR,
    env=environment,
    text=True,
    capture_output=True,
    check=True,
)
base_summary = json.loads((BASE_OUTPUT_DIR / "run_summary.json").read_text(encoding="utf-8"))
obsolete_keys = [key for key in base_summary if key.startswith("manual_")]
for key in obsolete_keys:
    base_summary.pop(key, None)
print(json.dumps(base_summary, indent=2))


{
  "status": "completed",
  "runtime_seconds": 457.9,
  "database_records": 9794932,
  "analysis_eligible_records": 9794932,
  "primary_soccer_cases": 170679,
  "primary_soccer_weighted_estimate": 5366681.015799999,
  "unique_week_rows_before_boundary_exclusion": 1410,
  "duplicate_week_keys": 0,
  "incomplete_boundary_weeks_excluded": 2,
  "model_weeks": 1408,
  "master_results_entries": 3947
}


In [3]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm


PROJECT_DIR = Path.cwd().resolve()
V2_DIR = PROJECT_DIR / "outputs" / "neiss_world_cup_v2"
CACHE_DIR = V2_DIR / "cache" / "yearly"
AUDIT_DIR = V2_DIR / "audit"
BASE_TABLE_DIR = V2_DIR / "tables"
OUT_DIR = PROJECT_DIR / "outputs" / "professor_revision_20260620"
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

Z95 = float(norm.ppf(0.975))
MEN_EVENTS = {
    2002: ("2002-05-31", "2002-06-30"),
    2006: ("2006-06-09", "2006-07-09"),
    2010: ("2010-06-11", "2010-07-11"),
    2014: ("2014-06-12", "2014-07-13"),
    2018: ("2018-06-14", "2018-07-15"),
    2022: ("2022-11-20", "2022-12-18"),
}
DOW_LABELS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def load_cached():
    psu_day = pd.concat(
        [pd.read_parquet(CACHE_DIR / f"psu_day_{year}.parquet") for year in range(1999, 2026)],
        ignore_index=True,
    )
    calendar = pd.read_csv(AUDIT_DIR / "world_cup_calendar_v2.csv", parse_dates=["date", "week_start"])
    psu_day = psu_day.merge(calendar, on="date", how="left", validate="many_to_one")
    return psu_day, calendar


def design_universe(psu_day: pd.DataFrame) -> pd.DataFrame:
    return psu_day[["strata_id", "psu_id"]].drop_duplicates().reset_index(drop=True)


def design_variance(frame: pd.DataFrame, value_col: str, universe: pd.DataFrame) -> float:
    values = (
        frame[["strata_id", "psu_id", value_col]]
        .groupby(["strata_id", "psu_id"], dropna=False)[value_col]
        .sum()
        .reset_index()
    )
    values = universe.merge(values, on=["strata_id", "psu_id"], how="left")
    values[value_col] = pd.to_numeric(values[value_col], errors="coerce").fillna(0.0)
    total_var = 0.0
    for _, group in values.groupby("strata_id", dropna=False):
        psu_totals = group.groupby("psu_id", dropna=False)[value_col].sum().to_numpy(float)
        m = len(psu_totals)
        if m < 2:
            continue
        total_var += (m / (m - 1)) * np.square(psu_totals - psu_totals.mean()).sum()
    return max(float(total_var), 0.0)


def psu_totals(psu_day: pd.DataFrame, mask: pd.Series, outcome: str, universe: pd.DataFrame) -> pd.DataFrame:
    cols = [f"{outcome}_w", f"{outcome}_n", "all_w", "all_n"]
    totals = (
        psu_day.loc[mask, ["strata_id", "psu_id", *cols]]
        .groupby(["strata_id", "psu_id"], dropna=False)[cols]
        .sum()
        .reset_index()
    )
    return universe.merge(totals, on=["strata_id", "psu_id"], how="left").fillna(0)


def period_stats(psu_totals_df: pd.DataFrame, outcome: str, universe: pd.DataFrame) -> dict:
    w_col, n_col = f"{outcome}_w", f"{outcome}_n"
    estimate = float(psu_totals_df[w_col].sum())
    n = int(psu_totals_df[n_col].sum())
    denominator_estimate = float(psu_totals_df["all_w"].sum())
    denominator_n = int(psu_totals_df["all_n"].sum())
    var = design_variance(psu_totals_df, w_col, universe)
    se = math.sqrt(var)
    cv = se / estimate if estimate else np.nan
    percentage = 100 * estimate / denominator_estimate if denominator_estimate else np.nan
    return {
        "unweighted_n": n,
        "weighted_estimate": estimate,
        "standard_error": se,
        "ci_lower": max(0.0, estimate - Z95 * se),
        "ci_upper": estimate + Z95 * se,
        "cv": cv,
        "weighted_percentage": percentage,
        "denominator_unweighted_n": denominator_n,
        "denominator_weighted_estimate": denominator_estimate,
    }


def compare_masks(
    psu_day: pd.DataFrame,
    calendar: pd.DataFrame,
    actual_mask: pd.Series,
    control_mask: pd.Series,
    actual_days: int,
    control_days: int,
    universe: pd.DataFrame,
    label: str,
    outcome: str = "soccer_product",
) -> dict:
    actual = psu_totals(psu_day, actual_mask, outcome, universe)
    control = psu_totals(psu_day, control_mask, outcome, universe)
    actual_stats = period_stats(actual, outcome, universe)
    control_stats = period_stats(control, outcome, universe)

    paired = actual[["strata_id", "psu_id", f"{outcome}_w", f"{outcome}_n", "all_w", "all_n"]].rename(
        columns={
            f"{outcome}_w": "actual_w",
            f"{outcome}_n": "actual_n",
            "all_w": "actual_all_w",
            "all_n": "actual_all_n",
        }
    ).merge(
        control[["strata_id", "psu_id", f"{outcome}_w", f"{outcome}_n", "all_w", "all_n"]].rename(
            columns={
                f"{outcome}_w": "control_w",
                f"{outcome}_n": "control_n",
                "all_w": "control_all_w",
                "all_n": "control_all_n",
            }
        ),
        on=["strata_id", "psu_id"],
        how="outer",
    ).fillna(0)

    paired["actual_daily"] = paired["actual_w"] / actual_days
    paired["control_daily"] = paired["control_w"] / control_days
    actual_daily = float(paired["actual_daily"].sum())
    control_daily = float(paired["control_daily"].sum())
    difference = actual_daily - control_daily
    paired["daily_difference_linearized"] = paired["actual_daily"] - paired["control_daily"]
    diff_se = math.sqrt(design_variance(paired, "daily_difference_linearized", universe))

    ratio = actual_daily / control_daily if control_daily else np.nan
    if actual_daily > 0 and control_daily > 0:
        paired["log_ratio_linearized"] = paired["actual_daily"] / actual_daily - paired["control_daily"] / control_daily
        log_ratio_se = math.sqrt(design_variance(paired, "log_ratio_linearized", universe))
        ratio_l = math.exp(math.log(ratio) - Z95 * log_ratio_se)
        ratio_u = math.exp(math.log(ratio) + Z95 * log_ratio_se)
    else:
        log_ratio_se = ratio_l = ratio_u = np.nan

    p_actual = actual_stats["weighted_estimate"] / actual_stats["denominator_weighted_estimate"]
    p_control = control_stats["weighted_estimate"] / control_stats["denominator_weighted_estimate"]
    pp_diff = 100 * (p_actual - p_control)
    paired["proportion_difference_linearized"] = 100 * (
        (paired["actual_w"] - p_actual * paired["actual_all_w"]) / actual_stats["denominator_weighted_estimate"]
        - (paired["control_w"] - p_control * paired["control_all_w"]) / control_stats["denominator_weighted_estimate"]
    )
    pp_diff_se = math.sqrt(design_variance(paired, "proportion_difference_linearized", universe))

    proportion_ratio = p_actual / p_control if p_control else np.nan
    paired["log_proportion_ratio_linearized"] = (
        paired["actual_w"] / actual_stats["weighted_estimate"]
        - paired["actual_all_w"] / actual_stats["denominator_weighted_estimate"]
        - paired["control_w"] / control_stats["weighted_estimate"]
        + paired["control_all_w"] / control_stats["denominator_weighted_estimate"]
    )
    log_prop_ratio_se = math.sqrt(design_variance(paired, "log_proportion_ratio_linearized", universe))

    actual_dates = calendar.loc[actual_mask.groupby(psu_day["date"]).any().reindex(calendar["date"], fill_value=False).to_numpy(), "date"]
    control_dates = calendar.loc[control_mask.groupby(psu_day["date"]).any().reindex(calendar["date"], fill_value=False).to_numpy(), "date"]

    row = {
        "analysis": label,
        "outcome": outcome,
        "actual_days": actual_days,
        "control_days": control_days,
        "actual_unweighted_n": actual_stats["unweighted_n"],
        "control_unweighted_n": control_stats["unweighted_n"],
        "actual_weighted_estimate": actual_stats["weighted_estimate"],
        "control_weighted_estimate": control_stats["weighted_estimate"],
        "actual_mean_daily_estimate": actual_daily,
        "control_mean_daily_estimate": control_daily,
        "absolute_mean_daily_difference": difference,
        "difference_ci_lower": difference - Z95 * diff_se,
        "difference_ci_upper": difference + Z95 * diff_se,
        "mean_daily_ratio": ratio,
        "ratio_ci_lower": ratio_l,
        "ratio_ci_upper": ratio_u,
        "actual_weighted_percentage": actual_stats["weighted_percentage"],
        "control_weighted_percentage": control_stats["weighted_percentage"],
        "percentage_point_difference": pp_diff,
        "percentage_point_difference_ci_lower": pp_diff - Z95 * pp_diff_se,
        "percentage_point_difference_ci_upper": pp_diff + Z95 * pp_diff_se,
        "weighted_percentage_ratio": proportion_ratio,
        "weighted_percentage_ratio_ci_lower": math.exp(math.log(proportion_ratio) - Z95 * log_prop_ratio_se)
        if proportion_ratio > 0 else np.nan,
        "weighted_percentage_ratio_ci_upper": math.exp(math.log(proportion_ratio) + Z95 * log_prop_ratio_se)
        if proportion_ratio > 0 else np.nan,
        "actual_denominator_weighted_estimate": actual_stats["denominator_weighted_estimate"],
        "control_denominator_weighted_estimate": control_stats["denominator_weighted_estimate"],
        "variance_method": "Direct with-replacement stratified PSU linearization of the paired contrast; survey-year strata and PSUs treated as distinct; no finite population correction.",
    }
    return row


def date_mask_from_dates(psu_day: pd.DataFrame, dates: set[pd.Timestamp]) -> pd.Series:
    normalized = {pd.Timestamp(d).normalize() for d in dates}
    return psu_day["date"].isin(normalized)


def make_same_dow_calendar(calendar: pd.DataFrame) -> pd.DataFrame:
    cal = calendar.copy()
    cal["same_dow_control"] = False
    cal["same_dow_match_set"] = pd.Series(pd.NA, index=cal.index, dtype="Int64")
    date_index = {pd.Timestamp(d).normalize(): idx for idx, d in enumerate(cal["date"])}
    any_tournament = dict(zip(cal["date"], cal["any_tournament"]))
    selected = []
    for tournament_year, (start_s, end_s) in MEN_EVENTS.items():
        for actual_date in pd.date_range(start_s, end_s, freq="D"):
            actual_dow = actual_date.dayofweek
            for control_year in [tournament_year - 1, tournament_year + 1]:
                if control_year < 1999 or control_year > 2025:
                    continue
                try:
                    base = pd.Timestamp(year=control_year, month=actual_date.month, day=actual_date.day)
                except ValueError:
                    continue
                candidates = []
                for shift in range(-3, 4):
                    candidate = (base + pd.Timedelta(days=shift)).normalize()
                    if candidate.year != control_year or candidate not in date_index:
                        continue
                    if candidate.dayofweek == actual_dow:
                        candidates.append((abs(shift), shift, candidate))
                if not candidates:
                    continue
                _, _, chosen = sorted(candidates)[0]
                if bool(any_tournament.get(chosen, False)):
                    continue
                selected.append((chosen, tournament_year))
    for chosen, tournament_year in selected:
        idx = date_index[chosen]
        cal.loc[idx, "same_dow_control"] = True
        cal.loc[idx, "same_dow_match_set"] = tournament_year
    return cal


def date_range_label(dates: pd.Series) -> str:
    if dates.empty:
        return ""
    parts = []
    for year, group in dates.sort_values().groupby(dates.dt.year):
        parts.append(f"{group.min().date()} to {group.max().date()}")
    return "; ".join(parts)


def model_matrix(weekly: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    t = (weekly["week_start"] - weekly["week_start"].min()).dt.days.to_numpy() / 365.25
    day = weekly["week_start"].dt.dayofyear.to_numpy()
    columns = {
        "Intercept": np.ones(len(weekly)),
        "men_tournament_fraction": weekly["men_tournament_fraction"].to_numpy(float),
        "matched_control_fraction": weekly["primary_control_fraction"].to_numpy(float),
        "year_trend": t,
        "covid_2020_2021": weekly["week_start"].dt.year.isin([2020, 2021]).to_numpy(float),
        "sin_annual_1": np.sin(2 * np.pi * day / 365.25),
        "cos_annual_1": np.cos(2 * np.pi * day / 365.25),
        "sin_annual_2": np.sin(4 * np.pi * day / 365.25),
        "cos_annual_2": np.cos(4 * np.pi * day / 365.25),
    }
    return np.column_stack(list(columns.values())), list(columns)


def hac_covariance(scores: np.ndarray, bread: np.ndarray, max_lags: int) -> np.ndarray:
    meat = scores.T @ scores
    n = len(scores)
    for lag in range(1, min(max_lags, n - 1) + 1):
        weight = 1 - lag / (max_lags + 1)
        cross = scores[lag:].T @ scores[:-lag]
        meat += weight * (cross + cross.T)
    return bread @ meat @ bread


def fit_ar1_lag_sensitivity(weekly: pd.DataFrame, lags: int) -> dict:
    work = weekly[(weekly["soccer_product_w"] > 0) & (weekly["soccer_product_variance"] > 0)].copy()
    X, names = model_matrix(work)
    y = np.log(work["soccer_product_w"].to_numpy(float))
    var_log = work["soccer_product_variance"].to_numpy(float) / np.square(work["soccer_product_w"].to_numpy(float))
    inv_var = 1 / np.clip(var_log, 1e-8, None)
    lo, hi = np.quantile(inv_var, [0.02, 0.98])
    w = np.clip(inv_var, lo, hi)
    beta = np.linalg.pinv(X.T @ (X * w[:, None])) @ (X.T @ (w * y))
    rho = 0.0
    for _ in range(20):
        residual = y - X @ beta
        denom = np.dot(residual[:-1], residual[:-1])
        rho_new = float(np.dot(residual[1:], residual[:-1]) / denom) if denom > 0 else 0.0
        rho_new = float(np.clip(rho_new, -0.95, 0.95))
        y_t = y[1:] - rho_new * y[:-1]
        X_t = X[1:] - rho_new * X[:-1]
        w_t = (w[1:] + w[:-1]) / 2
        bread = np.linalg.pinv(X_t.T @ (X_t * w_t[:, None]))
        beta_new = bread @ (X_t.T @ (w_t * y_t))
        if np.max(np.abs(beta_new - beta)) < 1e-9 and abs(rho_new - rho) < 1e-9:
            beta, rho = beta_new, rho_new
            break
        beta, rho = beta_new, rho_new

    y_t = y[1:] - rho * y[:-1]
    X_t = X[1:] - rho * X[:-1]
    w_t = (w[1:] + w[:-1]) / 2
    bread = np.linalg.pinv(X_t.T @ (X_t * w_t[:, None]))
    transformed_residual = y_t - X_t @ beta
    scores = X_t * (w_t * transformed_residual)[:, None]
    cov = hac_covariance(scores, bread, max_lags=lags)
    i_actual = names.index("men_tournament_fraction")
    i_control = names.index("matched_control_fraction")
    contrast = beta[i_actual] - beta[i_control]
    contrast_var = cov[i_actual, i_actual] + cov[i_control, i_control] - 2 * cov[i_actual, i_control]
    se = math.sqrt(max(float(contrast_var), 0.0))
    return {
        "hac_lags": lags,
        "rule": "Sensitivity around primary 8-lag Newey-West/HAC choice",
        "ar1_rho": rho,
        "coefficient": contrast,
        "robust_se": se,
        "adjusted_ratio": math.exp(contrast),
        "ci_lower": math.exp(contrast - Z95 * se),
        "ci_upper": math.exp(contrast + Z95 * se),
        "p_value": 2 * norm.sf(abs(contrast / se)) if se > 0 else np.nan,
        "n_weeks": len(work),
    }


def write_forest_plot(tournament_df: pd.DataFrame, primary_row: dict) -> None:
    rows = tournament_df[["tournament_year", "mean_daily_ratio", "ratio_ci_lower", "ratio_ci_upper"]].copy()
    rows["label"] = rows["tournament_year"].astype(str)
    pooled = pd.DataFrame([{
        "tournament_year": "Pooled",
        "mean_daily_ratio": primary_row["mean_daily_ratio"],
        "ratio_ci_lower": primary_row["ratio_ci_lower"],
        "ratio_ci_upper": primary_row["ratio_ci_upper"],
        "label": "Pooled",
    }])
    plot_df = pd.concat([rows, pooled], ignore_index=True)
    y = np.arange(len(plot_df))[::-1]
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    ax.axvline(1.0, color="black", linewidth=1.0, linestyle="--")
    for i, row in plot_df.iterrows():
        color = "black" if row["label"] == "Pooled" else "#4a4a4a"
        marker = "s" if row["label"] == "Pooled" else "o"
        ax.errorbar(
            row["mean_daily_ratio"],
            y[i],
            xerr=[[row["mean_daily_ratio"] - row["ratio_ci_lower"]], [row["ratio_ci_upper"] - row["mean_daily_ratio"]]],
            fmt=marker,
            color=color,
            ecolor=color,
            elinewidth=1.2,
            capsize=3,
            markersize=6,
        )
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["label"])
    ax.set_xlabel("Mean daily weighted estimate ratio (tournament / matched control)")
    ax.set_title("Men's FIFA World Cup tournament periods versus matched controls")
    ax.set_xlim(0.5, max(2.2, float(plot_df["ratio_ci_upper"].max()) * 1.08))
    ax.grid(axis="x", color="#d9d9d9", linewidth=0.6)
    ax.text(
        0.01,
        -0.18,
        "Ratios use NEISS product-code 1267 soccer-coded injuries; error bars are design-based 95% CIs.",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
    )
    fig.tight_layout()
    for ext in ["png", "svg", "pdf", "tiff"]:
        path = FIGURE_DIR / f"figure3_tournament_forest_plot.{ext}"
        if ext in {"png", "tiff"}:
            fig.savefig(path, dpi=600, bbox_inches="tight")
        else:
            fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def format_rounding_preview(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].round(3)
    return out


def main() -> None:
    psu_day, calendar = load_cached()
    universe = design_universe(psu_day)
    same_dow_calendar = make_same_dow_calendar(calendar)
    same_dow_map = same_dow_calendar[["date", "same_dow_control", "same_dow_match_set"]]
    psu_day = psu_day.drop(columns=["same_dow_control", "same_dow_match_set"], errors="ignore").merge(
        same_dow_map, on="date", how="left"
    )
    psu_day["same_dow_control"] = psu_day["same_dow_control"].fillna(False).astype(bool)

    primary = compare_masks(
        psu_day,
        calendar,
        psu_day["men_tournament"].fillna(False),
        psu_day["primary_control"].fillna(False),
        int(calendar["men_tournament"].sum()),
        int(calendar["primary_control"].sum()),
        universe,
        "Primary direct paired PSU contrast",
    )
    same_dow = compare_masks(
        psu_day,
        same_dow_calendar,
        psu_day["men_tournament"].fillna(False),
        psu_day["same_dow_control"].fillna(False),
        int(calendar["men_tournament"].sum()),
        int(same_dow_calendar["same_dow_control"].sum()),
        universe,
        "Same-day-of-week matched control sensitivity",
    )
    pre_2022 = compare_masks(
        psu_day,
        calendar,
        psu_day["men_tournament"].fillna(False) & psu_day["date"].dt.year.ne(2022),
        psu_day["primary_control"].fillna(False) & psu_day["match_set"].ne(2022),
        int(calendar.loc[calendar["men_tournament"] & calendar["year"].ne(2022), "date"].nunique()),
        int(calendar.loc[calendar["primary_control"] & calendar["match_set"].ne(2022), "date"].nunique()),
        universe,
        "Pre-2022-only sensitivity",
    )
    contrast_df = pd.DataFrame([primary, same_dow, pre_2022])
    contrast_df.to_csv(TABLE_DIR / "table_PR1_primary_and_key_sensitivity_contrasts.csv", index=False)

    tournament_rows = []
    for year in MEN_EVENTS:
        actual_cal = calendar["men_tournament"] & calendar["year"].eq(year)
        control_cal = calendar["primary_control"] & calendar["match_set"].eq(year)
        actual_mask = psu_day["men_tournament"].fillna(False) & psu_day["date"].dt.year.eq(year)
        control_mask = psu_day["primary_control"].fillna(False) & psu_day["match_set"].eq(year)
        row = compare_masks(
            psu_day,
            calendar,
            actual_mask,
            control_mask,
            int(calendar.loc[actual_cal, "date"].nunique()),
            int(calendar.loc[control_cal, "date"].nunique()),
            universe,
            f"{year} tournament-specific contrast",
        )
        candidate = calendar["primary_control_candidate"] & calendar["match_set"].eq(year)
        row.update({
            "tournament_year": year,
            "exposed_dates": date_range_label(calendar.loc[actual_cal, "date"]),
            "control_dates_retained": date_range_label(calendar.loc[control_cal, "date"]),
            "control_candidate_days": int(calendar.loc[candidate, "date"].nunique()),
            "control_dates_excluded": int(calendar.loc[candidate & ~calendar["primary_control"], "date"].nunique()),
        })
        tournament_rows.append(row)
    tournament_df = pd.DataFrame(tournament_rows)
    cols = [
        "tournament_year", "exposed_dates", "control_dates_retained", "actual_days", "control_candidate_days",
        "control_dates_excluded", "control_days", "actual_unweighted_n", "control_unweighted_n",
        "actual_weighted_estimate", "control_weighted_estimate", "actual_mean_daily_estimate",
        "control_mean_daily_estimate", "absolute_mean_daily_difference", "difference_ci_lower",
        "difference_ci_upper", "mean_daily_ratio", "ratio_ci_lower", "ratio_ci_upper",
        "actual_weighted_percentage", "control_weighted_percentage", "weighted_percentage_ratio",
        "weighted_percentage_ratio_ci_lower", "weighted_percentage_ratio_ci_upper", "variance_method",
    ]
    tournament_df = tournament_df[cols]
    tournament_df.to_csv(TABLE_DIR / "table_PR2_tournament_specific_contrasts.csv", index=False)

    loo_rows = []
    for year in MEN_EVENTS:
        row = compare_masks(
            psu_day,
            calendar,
            psu_day["men_tournament"].fillna(False) & psu_day["date"].dt.year.ne(year),
            psu_day["primary_control"].fillna(False) & psu_day["match_set"].ne(year),
            int(calendar.loc[calendar["men_tournament"] & calendar["year"].ne(year), "date"].nunique()),
            int(calendar.loc[calendar["primary_control"] & calendar["match_set"].ne(year), "date"].nunique()),
            universe,
            f"Leave out {year}",
        )
        row["excluded_tournament_year"] = year
        loo_rows.append(row)
    leave_one_out = pd.DataFrame(loo_rows)
    leave_one_out.to_csv(TABLE_DIR / "table_PR3_leave_one_tournament_out.csv", index=False)

    dow_rows = []
    for label, mask_col in [
        ("Men's tournament dates", "men_tournament"),
        ("Primary same-calendar controls", "primary_control"),
        ("Same-day-of-week controls", "same_dow_control"),
    ]:
        source = same_dow_calendar if mask_col == "same_dow_control" else calendar
        for dow, n in source.loc[source[mask_col].fillna(False), "day_of_week"].value_counts().sort_index().items():
            dow_rows.append({"group": label, "day_of_week": DOW_LABELS[int(dow)], "days": int(n)})
    dow_df = pd.DataFrame(dow_rows)
    dow_df.to_csv(TABLE_DIR / "table_PR4_day_of_week_balance.csv", index=False)

    weekly = pd.read_csv(AUDIT_DIR / "primary_model_observed_fitted.csv", parse_dates=["week_start"])
    lag_df = pd.DataFrame([fit_ar1_lag_sensitivity(weekly, lags) for lags in [4, 6, 8, 10, 12]])
    lag_df.to_csv(TABLE_DIR / "table_PR5_newey_west_lag_sensitivity.csv", index=False)

    write_forest_plot(tournament_df, primary)

    old_master = pd.read_csv(PROJECT_DIR / "outputs" / "master_results" / "master_results_dictionary.csv")
    add_rows = []
    for table_name, frame in {
        "table_PR1_primary_and_key_sensitivity_contrasts": contrast_df,
        "table_PR2_tournament_specific_contrasts": tournament_df,
        "table_PR3_leave_one_tournament_out": leave_one_out,
        "table_PR4_day_of_week_balance": dow_df,
        "table_PR5_newey_west_lag_sensitivity": lag_df,
    }.items():
        source = TABLE_DIR / f"{table_name}.csv"
        for i, row in frame.reset_index(drop=True).iterrows():
            for col, value in row.items():
                if isinstance(value, (int, float, np.integer, np.floating)) and pd.notna(value):
                    add_rows.append({
                        "result_id": f"{table_name}.r{i + 1}.{col}",
                        "source_table": table_name,
                        "source_file": str(source),
                        "source_row": i + 2,
                        "metric": col,
                        "value": value,
                        "ci_lower": "",
                        "ci_upper": "",
                        "numerator_definition": "NEISS product code 1267 soccer-coded injuries unless otherwise specified",
                        "denominator_definition": "NEISS-estimated emergency department-treated consumer product and recreation-related injuries",
                        "outcome_definition": "Soccer-coded NEISS injury burden",
                        "exposure_group": str(row.get("analysis", row.get("tournament_year", ""))),
                        "comparison_group": "Matched controls",
                        "stability_flag": "",
                        "verification_status": "Professor revision addendum generated from cached v2 PSU-day data",
                    })
    master = pd.concat([old_master, pd.DataFrame(add_rows)], ignore_index=True)
    master.to_csv(OUT_DIR / "master_results_dictionary_with_professor_revision_addendum.csv", index=False)

    summary_lines = [
        "# Professor Revision Analysis Addendum",
        "",
        f"Primary direct paired PSU ratio: {primary['mean_daily_ratio']:.3f} ({primary['ratio_ci_lower']:.3f} to {primary['ratio_ci_upper']:.3f})",
        f"Primary absolute daily difference: {primary['absolute_mean_daily_difference']:.1f} ({primary['difference_ci_lower']:.1f} to {primary['difference_ci_upper']:.1f})",
        f"Applied to {primary['actual_days']} tournament days: {primary['absolute_mean_daily_difference'] * primary['actual_days']:.0f} additional NEISS-estimated ED-treated soccer-coded injuries.",
        f"Same-day-of-week control ratio: {same_dow['mean_daily_ratio']:.3f} ({same_dow['ratio_ci_lower']:.3f} to {same_dow['ratio_ci_upper']:.3f})",
        f"Pre-2022-only ratio: {pre_2022['mean_daily_ratio']:.3f} ({pre_2022['ratio_ci_lower']:.3f} to {pre_2022['ratio_ci_upper']:.3f})",
        "",
        "Generated files:",
    ]
    for path in sorted(TABLE_DIR.glob("*.csv")) + sorted(FIGURE_DIR.glob("*")):
        summary_lines.append(f"- {path.relative_to(PROJECT_DIR)}")
    (OUT_DIR / "README.md").write_text("\n".join(summary_lines), encoding="utf-8")

    print("\n".join(summary_lines[:8]))
    print(f"Output directory: {OUT_DIR}")


if __name__ == "__main__":
    main()


# Professor Revision Analysis Addendum

Primary direct paired PSU ratio: 1.179 (1.002 to 1.388)
Primary absolute daily difference: 68.9 (-0.5 to 138.3)
Applied to 186 tournament days: 12823 additional NEISS-estimated ED-treated soccer-coded injuries.
Same-day-of-week control ratio: 1.166 (0.991 to 1.372)
Pre-2022-only ratio: 1.221 (1.035 to 1.441)

Output directory: D:\ai-job-agent\MIMIC_Project_3\outputs\professor_revision_20260620


In [4]:
CPSC_ROWS = [
    {"year": 2021, "cpsc_weighted_estimate": 144895, "cpsc_unweighted_cases": 5611, "cpsc_cv": 0.14, "cpsc_ci_lower": 104808, "cpsc_ci_upper": 184982},
    {"year": 2022, "cpsc_weighted_estimate": 179284, "cpsc_unweighted_cases": 6308, "cpsc_cv": 0.15, "cpsc_ci_lower": 125452, "cpsc_ci_upper": 233115},
    {"year": 2023, "cpsc_weighted_estimate": 212423, "cpsc_unweighted_cases": 7603, "cpsc_cv": 0.14, "cpsc_ci_lower": 153084, "cpsc_ci_upper": 271761},
    {"year": 2024, "cpsc_weighted_estimate": 265761, "cpsc_unweighted_cases": 8356, "cpsc_cv": 0.11, "cpsc_ci_lower": 206788, "cpsc_ci_upper": 324735},
    {"year": 2025, "cpsc_weighted_estimate": 264199, "cpsc_unweighted_cases": 8951, "cpsc_cv": 0.13, "cpsc_ci_lower": 198335, "cpsc_ci_upper": 330062},
]

local_burden = pd.read_csv(BASE_OUTPUT_DIR / "tables" / "table_B_primary_soccer_injury_burden.csv")
local_by_year = local_burden.set_index("year")
calibration_rows = []
for benchmark in CPSC_ROWS:
    year = benchmark["year"]
    local = local_by_year.loc[year]
    local_estimate = float(local["weighted_estimate"])
    local_n = int(local["unweighted_n"])
    calibration_rows.append(
        {
            "year": year,
            "benchmark_source": "CPSC NEISS Query Builder",
            "query_timestamp": "2026-06-17 17:38:16",
            "query_selection": "Most Recent 5 Years (2021-2025); Product Selection: Soccer (activity, Apparel Or Equipment) (1267); grouped by YEAR",
            "cpsc_unweighted_cases": benchmark["cpsc_unweighted_cases"],
            "local_unweighted_n": local_n,
            "unweighted_match": local_n == benchmark["cpsc_unweighted_cases"],
            "cpsc_weighted_estimate": benchmark["cpsc_weighted_estimate"],
            "local_weighted_estimate": local_estimate,
            "weighted_estimate_rounded": round(local_estimate),
            "estimate_abs_diff": local_estimate - benchmark["cpsc_weighted_estimate"],
            "estimate_relative_diff_pct": 100 * (local_estimate - benchmark["cpsc_weighted_estimate"]) / benchmark["cpsc_weighted_estimate"],
            "estimate_match_after_rounding": round(local_estimate) == benchmark["cpsc_weighted_estimate"],
            "cpsc_cv_displayed": benchmark["cpsc_cv"],
            "local_cv": float(local["cv"]),
            "cpsc_ci_lower": benchmark["cpsc_ci_lower"],
            "local_ci_lower": float(local["ci_lower"]),
            "ci_lower_abs_diff": float(local["ci_lower"]) - benchmark["cpsc_ci_lower"],
            "cpsc_ci_upper": benchmark["cpsc_ci_upper"],
            "local_ci_upper": float(local["ci_upper"]),
            "ci_upper_abs_diff": float(local["ci_upper"]) - benchmark["cpsc_ci_upper"],
            "calibration_interpretation": "Local unweighted case count and weighted national estimate reproduce the CPSC Query Builder output after rounding; confidence limits differ slightly from the local z-based public-use variance implementation.",
        }
    )

calibration = pd.DataFrame(calibration_rows)
calibration_path = REVISION_OUTPUT_DIR / "tables" / "table_PR6_cpsc_calibration_benchmark.csv"
calibration.to_csv(calibration_path, index=False)
display(calibration)


   year          benchmark_source      query_timestamp                                                                                                      query_selection  cpsc_unweighted_cases  local_unweighted_n  unweighted_match  cpsc_weighted_estimate  local_weighted_estimate  weighted_estimate_rounded  estimate_abs_diff  estimate_relative_diff_pct  estimate_match_after_rounding  cpsc_cv_displayed  local_cv  cpsc_ci_lower  local_ci_lower  ci_lower_abs_diff  cpsc_ci_upper  local_ci_upper  ci_upper_abs_diff                                                                                                                                                                                       calibration_interpretation
0  2021  CPSC NEISS Query Builder  2026-06-17 17:38:16  Most Recent 5 Years (2021-2025); Product Selection: Soccer (activity, Apparel Or Equipment) (1267); grouped by YEAR                   5611                5611              True                  144895              14489

In [5]:
table_dir = REVISION_OUTPUT_DIR / "tables"
figure_dir = REVISION_OUTPUT_DIR / "figures"
expected_tables = [
    "table_PR1_primary_and_key_sensitivity_contrasts.csv",
    "table_PR2_tournament_specific_contrasts.csv",
    "table_PR3_leave_one_tournament_out.csv",
    "table_PR4_day_of_week_balance.csv",
    "table_PR5_newey_west_lag_sensitivity.csv",
    "table_PR6_cpsc_calibration_benchmark.csv",
]
expected_figures = [
    "figure3_tournament_forest_plot.png",
    "figure3_tournament_forest_plot.svg",
    "figure3_tournament_forest_plot.pdf",
    "figure3_tournament_forest_plot.tiff",
]

missing_outputs = [str(table_dir / name) for name in expected_tables if not (table_dir / name).exists()]
missing_outputs += [str(figure_dir / name) for name in expected_figures if not (figure_dir / name).exists()]
if missing_outputs:
    raise FileNotFoundError(f"Missing required revision outputs: {missing_outputs}")

primary_results = pd.read_csv(table_dir / expected_tables[0])
primary = primary_results.loc[primary_results["analysis"].eq("Primary direct paired PSU contrast")].iloc[0]
same_dow = primary_results.loc[primary_results["analysis"].eq("Same-day-of-week matched control sensitivity")].iloc[0]
pre_2022 = primary_results.loc[primary_results["analysis"].eq("Pre-2022-only sensitivity")].iloc[0]
tournament_results = pd.read_csv(table_dir / expected_tables[1])
leave_one_out_results = pd.read_csv(table_dir / expected_tables[2])
hac_results = pd.read_csv(table_dir / expected_tables[4])
calibration_results = pd.read_csv(table_dir / expected_tables[5])

if int(primary["actual_days"]) != 186 or int(primary["control_days"]) != 308:
    raise AssertionError("Primary exposure or comparison day count changed unexpectedly")
if len(tournament_results) != 6:
    raise AssertionError("Tournament-specific analysis must include six men's World Cups")
if len(leave_one_out_results) != 6:
    raise AssertionError("Leave-one-tournament-out analysis must include six exclusions")
if set(hac_results["hac_lags"].astype(int)) != {4, 6, 8, 10, 12}:
    raise AssertionError("HAC lag sensitivity is incomplete")
if not calibration_results["unweighted_match"].astype(bool).all():
    raise AssertionError("CPSC calibration case counts do not match")
if not calibration_results["estimate_match_after_rounding"].astype(bool).all():
    raise AssertionError("CPSC calibration weighted estimates do not match after rounding")

prior_dir = PROJECT_DIR / "outputs" / "professor_revision_20260617" / "tables"
reproduction = {}
for name in expected_tables[:5]:
    current = pd.read_csv(table_dir / name)
    prior_path = prior_dir / name
    if prior_path.exists():
        prior = pd.read_csv(prior_path)
        numeric_columns = sorted(set(current.select_dtypes(include=[np.number]).columns) & set(prior.select_dtypes(include=[np.number]).columns))
        aligned_rows = min(len(current), len(prior))
        maximum_difference = 0.0
        if numeric_columns and aligned_rows:
            delta = np.abs(current.loc[:aligned_rows - 1, numeric_columns].to_numpy(float) - prior.loc[:aligned_rows - 1, numeric_columns].to_numpy(float))
            maximum_difference = float(np.nanmax(delta)) if delta.size else 0.0
        reproduction[name] = {"rows": len(current), "prior_rows": len(prior), "maximum_numeric_absolute_difference": maximum_difference}

master_base = pd.read_csv(PROJECT_DIR / "outputs" / "master_results" / "master_results_dictionary.csv")
addendum_rows = []
for table_name in expected_tables:
    frame = pd.read_csv(table_dir / table_name)
    for row_index, row in frame.reset_index(drop=True).iterrows():
        for metric, value in row.items():
            if isinstance(value, (int, float, np.integer, np.floating)) and pd.notna(value):
                addendum_rows.append(
                    {
                        "result_id": f"{Path(table_name).stem}.r{row_index + 1}.{metric}",
                        "source_table": Path(table_name).stem,
                        "source_file": str(table_dir / table_name),
                        "source_row": row_index + 2,
                        "metric": metric,
                        "value": value,
                        "ci_lower": "",
                        "ci_upper": "",
                        "numerator_definition": "NEISS product code 1267 soccer-coded injuries unless otherwise specified",
                        "denominator_definition": "NEISS-estimated emergency department-treated consumer product and recreation-related injuries",
                        "outcome_definition": "Soccer-coded NEISS injury burden",
                        "exposure_group": str(row.get("analysis", row.get("tournament_year", row.get("year", "")))),
                        "comparison_group": "Matched same-calendar control dates unless otherwise specified",
                        "stability_flag": "",
                        "verification_status": "Regenerated and verified in the v3 executed notebook",
                    }
                )

verified_master = pd.concat([master_base, pd.DataFrame(addendum_rows)], ignore_index=True)
master_path = REVISION_OUTPUT_DIR / "master_results_dictionary_verified_20260620.csv"
verified_master.to_csv(master_path, index=False)

verification = {
    "annual_workbooks_verified": len(RAW_FILES),
    "yearly_psu_day_caches_verified": len(CACHE_FILES),
    "required_tables_verified": expected_tables,
    "required_figures_verified": expected_figures,
    "primary_mean_daily_ratio": float(primary["mean_daily_ratio"]),
    "primary_ratio_ci": [float(primary["ratio_ci_lower"]), float(primary["ratio_ci_upper"])],
    "primary_absolute_mean_daily_difference": float(primary["absolute_mean_daily_difference"]),
    "primary_difference_ci": [float(primary["difference_ci_lower"]), float(primary["difference_ci_upper"])],
    "estimated_additional_injuries_over_186_tournament_days": float(primary["absolute_mean_daily_difference"] * primary["actual_days"]),
    "same_day_of_week_ratio": float(same_dow["mean_daily_ratio"]),
    "pre_2022_ratio": float(pre_2022["mean_daily_ratio"]),
    "cpsc_case_count_matches": bool(calibration_results["unweighted_match"].astype(bool).all()),
    "cpsc_rounded_estimate_matches": bool(calibration_results["estimate_match_after_rounding"].astype(bool).all()),
    "cpsc_confidence_interval_note": "Official CPSC and local confidence limits are both retained because the local public-use variance implementation differs slightly from the Query Builder.",
    "reproduction_against_20260617": reproduction,
    "master_results_file": str(master_path),
}
verification_path = REVISION_OUTPUT_DIR / "verification_report.json"
verification_path.write_text(json.dumps(verification, indent=2), encoding="utf-8")

checksums = {}
for path in sorted(list(table_dir.glob("*.csv")) + list(figure_dir.glob("*")) + [master_path, verification_path]):
    checksums[str(path.relative_to(PROJECT_DIR))] = hashlib.sha256(path.read_bytes()).hexdigest()
(REVISION_OUTPUT_DIR / "sha256_manifest.json").write_text(json.dumps(checksums, indent=2), encoding="utf-8")

print(json.dumps(verification, indent=2))


{
  "annual_workbooks_verified": 27,
  "yearly_psu_day_caches_verified": 27,
  "required_tables_verified": [
    "table_PR1_primary_and_key_sensitivity_contrasts.csv",
    "table_PR2_tournament_specific_contrasts.csv",
    "table_PR3_leave_one_tournament_out.csv",
    "table_PR4_day_of_week_balance.csv",
    "table_PR5_newey_west_lag_sensitivity.csv",
    "table_PR6_cpsc_calibration_benchmark.csv"
  ],
  "required_figures_verified": [
    "figure3_tournament_forest_plot.png",
    "figure3_tournament_forest_plot.svg",
    "figure3_tournament_forest_plot.pdf",
    "figure3_tournament_forest_plot.tiff"
  ],
  "primary_mean_daily_ratio": 1.1794828825242856,
  "primary_ratio_ci": [
    1.0020415674943612,
    1.3883454691869617
  ],
  "primary_absolute_mean_daily_difference": 68.94110543918458,
  "primary_difference_ci": [
    -0.4581370595042955,
    138.34034793787345
  ],
  "estimated_additional_injuries_over_186_tournament_days": 12823.045611688332,
  "same_day_of_week_ratio": 1.1659170

In [6]:
primary_display = primary_results[
    [
        "analysis",
        "actual_days",
        "control_days",
        "actual_unweighted_n",
        "control_unweighted_n",
        "actual_weighted_estimate",
        "control_weighted_estimate",
        "absolute_mean_daily_difference",
        "difference_ci_lower",
        "difference_ci_upper",
        "mean_daily_ratio",
        "ratio_ci_lower",
        "ratio_ci_upper",
        "actual_weighted_percentage",
        "control_weighted_percentage",
        "weighted_percentage_ratio",
        "weighted_percentage_ratio_ci_lower",
        "weighted_percentage_ratio_ci_upper",
    ]
].copy()
display(primary_display.round(3))
display(tournament_results[["tournament_year", "actual_days", "control_days", "actual_unweighted_n", "control_unweighted_n", "mean_daily_ratio", "ratio_ci_lower", "ratio_ci_upper"]].round(3))
display(hac_results.round(4))
print(f"Primary paired-PSU ratio: {primary['mean_daily_ratio']:.3f} ({primary['ratio_ci_lower']:.3f} to {primary['ratio_ci_upper']:.3f})")
print(f"Primary daily difference: {primary['absolute_mean_daily_difference']:.1f} ({primary['difference_ci_lower']:.1f} to {primary['difference_ci_upper']:.1f})")
print(f"Estimated difference across 186 tournament days: {primary['absolute_mean_daily_difference'] * primary['actual_days']:.0f}")
print(f"Same-day-of-week sensitivity ratio: {same_dow['mean_daily_ratio']:.3f} ({same_dow['ratio_ci_lower']:.3f} to {same_dow['ratio_ci_upper']:.3f})")
print(f"Pre-2022 sensitivity ratio: {pre_2022['mean_daily_ratio']:.3f} ({pre_2022['ratio_ci_lower']:.3f} to {pre_2022['ratio_ci_upper']:.3f})")
print(f"Verified master results: {master_path}")


                                       analysis  actual_days  control_days  actual_unweighted_n  control_unweighted_n  actual_weighted_estimate  control_weighted_estimate  absolute_mean_daily_difference  difference_ci_lower  difference_ci_upper  mean_daily_ratio  ratio_ci_lower  ratio_ci_upper  actual_weighted_percentage  control_weighted_percentage  weighted_percentage_ratio  weighted_percentage_ratio_ci_lower  weighted_percentage_ratio_ci_upper
0            Primary direct paired PSU contrast          186           308                 2656                  3804                 84267.439                 118305.769                          68.941               -0.458              138.340             1.179           1.002           1.388                       1.148                        0.991                      1.159                               1.009                               1.331
1  Same-day-of-week matched control sensitivity          186           307                 2656   

In [7]:
quasi_poisson_path = BASE_OUTPUT_DIR / "tables" / "table_E_primary_model_results.csv"
model_results = pd.read_csv(quasi_poisson_path)
quasi_poisson_result = model_results[
    model_results["model"].eq("Unweighted sampled-case quasi-Poisson/HAC sensitivity")
    & model_results["term"].eq("men_tournament_vs_matched_control")
][
    [
        "model",
        "term",
        "adjusted_ratio",
        "ci_lower",
        "ci_upper",
        "p_value",
        "pearson_dispersion",
        "n_weeks",
        "model_family",
        "numerator_definition",
        "denominator_definition",
        "comparison_group",
    ]
].copy()
if len(quasi_poisson_result) != 1:
    raise AssertionError("Expected one quasi-Poisson tournament-versus-control result")
display(quasi_poisson_result.round(4))
print(f"Source table: {quasi_poisson_path}")


                                                    model                               term  adjusted_ratio  ci_lower  ci_upper  p_value  pearson_dispersion  n_weeks                                                   model_family                           numerator_definition                                                                   denominator_definition                                     comparison_group
91  Unweighted sampled-case quasi-Poisson/HAC sensitivity  men_tournament_vs_matched_control          1.1463    1.0058    1.3065   0.0408              5.8424     1408  Poisson mean model with HAC sandwich; overdispersion reported  Product-code-1267 soccer injury weekly burden  Unweighted sampled NEISS consumer product and recreation-related records used as offset  Matched same-calendar controls for primary contrast
Source table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_v2\tables\table_E_primary_model_results.csv
